# Add-it — Object Insertion on SD3.5 Medium

Object insertion inspired by **Add-it** (Tewel et al., ICLR 2025, arXiv:2411.07232),
on **SD3.5 Medium**. Two modes (`ADDIT_MODE` in `addit_config`):

**`composite` (default)** — keeps Add-it's affordance thesis (no input bbox, the model decides
*where*) but enforces the repo's core rule that the source image is the trusted background:
1. **img2img** a person into the scene (the model places it).
2. **YOLOv8-seg** the *added* person (filtered against people already in the source).
3. **Composite** only those pixels onto the byte-exact source → background preserved 100%, output sharp.

**`faithful`** — the paper's training-free attention-injection method (weighted extended
self-attention eq. 3 + auto-γ root-solver, structure transfer §3.3, subject-guided latent
blending §3.4 with SAM-2, no inversion §3.5). Re-noises and decodes the whole image, so the
background passes through the VAE. Kept for reference / ablation.

See `docs/addit_paper_fidelity.md` for the equation-by-equation mapping of the faithful mode.

## 1. Install dependencies

In [ ]:
!pip -q install -U diffusers transformers accelerate safetensors huggingface_hub scikit-image scipy ultralytics
# SAM-2 is a SOFT dependency: if this install fails, Add-it falls back to the Otsu coarse mask.
!pip -q install -U "git+https://github.com/facebookresearch/sam2.git" || echo "sam2 install skipped; will use transformers Sam2 or Otsu fallback" 

## 2. Clone or update repo

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/BDT-17/VIN.git"
REPO_DIR = Path("/kaggle/working/VIN")

if REPO_DIR.exists():
    %cd /kaggle/working/VIN
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd /kaggle/working/VIN

PROJECT_DIR = REPO_DIR
ADDIT_DIR = PROJECT_DIR / "addit(experimental)"
if not (PROJECT_DIR / "sd35_config.py").exists() or not ADDIT_DIR.exists():
    raise FileNotFoundError(f"Expected repo root with sd35_config.py and addit(experimental)/: {PROJECT_DIR}")
%cd {PROJECT_DIR}
!git log --oneline -3

## 3. Imports

In [ ]:
import os, sys
from pathlib import Path
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_DIR = Path("/kaggle/working/VIN") if Path("/kaggle/working/VIN/addit(experimental)").exists() else Path.cwd()
ADDIT_DIR = PROJECT_DIR / "addit(experimental)"
%cd {PROJECT_DIR}
for d in (PROJECT_DIR, ADDIT_DIR):
    d = str(d)
    if d in sys.path:
        sys.path.remove(d)
    sys.path.insert(0, d)
for m in ("addit_config", "addit_core", "addit_sam", "addit_pipeline"):
    sys.modules.pop(m, None)

from sd35_config import RESOLUTION, TRAIN_DEVICE
from sd35_model import build_img2img_pipeline
from sd35_utils import clear_cuda
import addit_config as C
from addit_pipeline import AddItPipeline, save_addit_debug
from addit_sam import sam2_available

print("Imports OK")
print("Backbone:", C.ADDIT_BACKBONE, "| steps:", C.ADDIT_NUM_INFERENCE_STEPS, "| gamma mode:", C.ADDIT_GAMMA_MODE)
print("t_struct(real/gen):", C.ADDIT_T_STRUCT_FRAC_REAL, C.ADDIT_T_STRUCT_FRAC_GEN, "| t_blend:", C.ADDIT_T_BLEND_FRAC)

## 4. Runtime check

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(i, p.name, round(p.total_memory/1024**3, 2), "GB")
print("Resolution:", RESOLUTION)
print("SAM-2 available:", sam2_available())   # False -> Otsu fallback (paper-ablation: 0.809 vs 0.828)

## 5. Hugging Face login (SD3.5 is gated)

In [ ]:
import os
from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = hf_token or UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("Kaggle secret lookup skipped:", type(e).__name__)
if hf_token:
    login(token=hf_token); print("HF login OK")
else:
    print("No HF_TOKEN; continue only if SD3.5 is already cached/accessible.")

## 6. Load SD3.5 pipeline
We use the img2img pipeline only as a convenient container for the VAE / transformer / scheduler / text encoders; Add-it drives its own denoise loop.

In [ ]:
device = C.ADDIT_PRIMARY_DEVICE if torch.cuda.is_available() else "cpu"
pipe = build_img2img_pipeline(device=device)
addit = AddItPipeline(pipe, device=device)
print("Add-it pipeline ready on", addit.device)
print("MM-DiT blocks:", addit.num_layers,
      "| inject layers:", sorted(addit.state.inject_layers)[:6], "...",
      "| mask layers:", sorted(addit.state.mask_layers))

## 7. Demo — add an object (model decides where)
Give a **target prompt** describing the edited image and the **subject token** (the noun for the added object). No bbox — the model decides where.

Default `ADDIT_MODE="composite"`: img2img → YOLO-seg the added person → paste onto the byte-exact source (background preserved 100%, output sharp). Set `C.ADDIT_MODE="faithful"` before building the pipeline to run the paper's attention-injection loop instead.

In [ ]:
from PIL import Image

# Use any source image as the trusted background. For a self-contained demo we
# synthesise an empty street; in practice load a real frame:
#   source = Image.open("/kaggle/input/.../frame.jpg").convert("RGB")
src_gen = pipe(
    prompt="a photo of an empty city street, daytime, no people",
    image=Image.new("RGB", (RESOLUTION, RESOLUTION), (127,127,127)),
    strength=1.0, num_inference_steps=28, guidance_scale=5.0,
).images[0].resize((RESOLUTION, RESOLUTION))

# Composite mode (ADDIT_MODE="composite"): the model decides WHERE to add the
# person (no bbox); we segment only the added person and paste it onto the
# byte-exact source, so the background is preserved 100% and stays sharp.
res = addit.add_object(
    src_gen,
    target_prompt="a photo of a city street with a pedestrian walking on the sidewalk",
    subject_token="pedestrian",
    seed=C.ADDIT_SEED,
)
print("success:", res.success,
      "| person added:", res.mask_image is not None,
      "| note:", res.error or "ok")
save_addit_debug(res, tag="demo")
display(res.source_image); display(res.result_image)
if res.mask_image is not None: display(res.mask_image)


## 8. Real source images (CityPersons / any dataset)
Real photos use the no-inversion re-noising (§3.5) and `t_struct=867`.

In [ ]:
# Point this at any real images you have mounted (the trusted backgrounds).
SOURCE_GLOBS = [
    "/kaggle/input/**/*.jpg", "/kaggle/input/**/*.png",
]
import glob
real_paths = []
for g in SOURCE_GLOBS:
    real_paths += glob.glob(g, recursive=True)
real_paths = sorted(real_paths)[:5]
print("Found", len(real_paths), "real source images")

EDITS = [
    ("a photo of a city street with a pedestrian walking", "pedestrian"),
    ("an urban sidewalk with a person standing near the curb", "person"),
]
results = []
for k, p in enumerate(real_paths):
    src = Image.open(p).convert("RGB")
    prompt, subj = EDITS[k % len(EDITS)]
    r = addit.add_object(src, target_prompt=prompt, subject_token=subj,
                         seed=C.ADDIT_SEED + k)
    save_addit_debug(r, tag=f"real{k}")
    results.append(r)
    print(f"[{k}] {Path(p).name}: success={r.success} added={r.mask_image is not None} note={r.error or 'ok'}")
    if r.result_image is not None:
        display(r.result_image)
clear_cuda()


## 9. Step-by-step scene building (paper §3.5, fig. 10)
Each edit operates on the previous output, progressively composing a scene.

In [ ]:
seq = addit.step_by_step(
    src_gen,
    edits=[
        ("a photo of a city street with a pedestrian walking", "pedestrian"),
        ("a city street with a pedestrian walking and a dog beside them", "dog"),
    ],
    seed=C.ADDIT_SEED,
)
for j, r in enumerate(seq):
    print(f"edit {j}: success={r.success}")
    if r.result_image is not None: display(r.result_image)
clear_cuda()

## 10. Shared metrics (comparable with V5 + LoRA flows)
Computes the cross-flow schema — person detection, **Inclusion** (was an object actually added),
scale, and background preservation — via the root `shared_metrics.py`. The same schema is emitted
by the V5 (`sd35_metrics.compute_shared_case_metrics`) and LoRA
(`LoRA.inference.inpaint_metrics.compute_shared_case_metrics`) flows, so the three can be compared
head-to-head. The YOLO detector is injected (soft): without it, detection-based metrics are skipped.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from addit_metrics import make_yolo_detector, metrics_for_result
from shared_metrics import summarize_shared

# Injected detector (soft): if ultralytics/YOLO is unavailable, pass detector=None.
try:
    detector = make_yolo_detector("yolov8m-seg.pt", device=device)
except Exception as e:
    print("YOLO detector unavailable:", type(e).__name__, "-> detection metrics skipped")
    detector = None

# Optional expected person height in px for scale_error; None -> scale stays None.
EXPECTED_HEIGHT = None

# Pair each AddItResult with its SOURCE identity so case_id matches the V5 / LoRA
# flows (paired_shared keys on (case_id, seed)). Cell 8 builds `results` in the
# same order as `real_paths`; the demo (cell 7) has no real source -> "demo".
eval_items = []
if "res" in dir():
    eval_items.append(("demo", getattr(res, "seed", C.ADDIT_SEED), res))
if "results" in dir():
    for k, r in enumerate(results):
        cid = Path(real_paths[k]).stem if ("real_paths" in dir() and k < len(real_paths)) else f"case{k}"
        eval_items.append((cid, getattr(r, "seed", C.ADDIT_SEED + k), r))

rows = []
for cid, seed, r in eval_items:
    m = metrics_for_result(r, detector=detector, expected_height=EXPECTED_HEIGHT)
    rows.append({"flow": "addit", "case_id": cid, "seed": seed, **m})

addit_shared_df = pd.DataFrame(rows)
out_csv = "/kaggle/working/addit_shared_metrics.csv"
addit_shared_df.to_csv(out_csv, index=False)
print(f"Add-it shared metrics: {len(rows)} rows -> {out_csv}")
print(json.dumps(summarize_shared(rows), indent=2, ensure_ascii=False))
addit_shared_df.head()

## Notes on fidelity
- The **mechanism** matches the paper exactly (extended attention eq. 3, auto-γ root-solver,
  structure transfer, subject-attention → Otsu → SAM-2 → latent blend, no inversion).
- The paper's FLUX-specific integer timesteps and layer indices are re-expressed as **fractions**
  of SD3.5's schedule/24-block stack — see `docs/addit_paper_fidelity.md`.
- SD3.5 needs real CFG (FLUX-dev used distilled guidance), so a CFG scale and light negative prompt
  are used; this is the one unavoidable backbone difference.